# Week 10: Advanced Simulation with ASE and CHGNet

In this week's lab, we will combine the **Atomic Simulation Environment (ASE)** with **CHGNet** (a universal graph neural network potential for materials) to perform practical, high-level atomistic simulations.

We will cover two main applications:
1. **Adsorption Energy Calculation**: How strongly does a molecule bind to a surface?
2. **Diffusion Coefficient via Molecular Dynamics (MD)**: How fast do atoms move in a material at a given temperature?

### Environment Setup
First, let's make sure we have the necessary libraries installed.

In [ ]:
!pip install -q ase chgnet pymatgen matplotlib mp-api

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ase import Atoms
from ase.build import fcc111, molecule, add_adsorbate, bulk
from ase.optimize import FIRE
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.visualize.plot import plot_atoms
from ase import units

from chgnet.model.dynamics import CHGNetCalculator

## Part 1: Adsorption Energy Calculation

Adsorption is a critical process in catalysis, gas sensors, and energy storage. The adsorption energy ($E_{ads}$) tells us whether the process is energetically favorable.

It is calculated as:
$$E_{ads} = E_{system} - (E_{surface} + E_{adsorbate})$$

Where:
- $E_{system}$: Energy of the surface with the adsorbate attached.
- $E_{surface}$: Energy of the bare surface.
- $E_{adsorbate}$: Energy of the isolated molecule.

A negative $E_{ads}$ means the adsorption is exothermic (favorable).

### 1.1 Building the Systems

In [ ]:
# Initialize CHGNet Calculator
calculator = CHGNetCalculator()

# 1. Build the Pt(111) surface (slab)
# We create a 3x3x3 supercell of the fcc(111) surface
slab = fcc111('Pt', size=(3, 3, 3), vacuum=10.0)
slab.calc = calculator

# 2. Build the adsorbate (CO molecule)
adsorbate = molecule('CO')
# Add vacuum box around the molecule for calculation
adsorbate.set_cell([15, 15, 15])
adsorbate.center()
adsorbate.calc = calculator

In [ ]:
# Visualize the initial structures
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
plot_atoms(slab, ax[0], rotation=('-75x, 45y, 10z'))
ax[0].set_title('Pt(111) Slab')
plot_atoms(adsorbate, ax[1])
ax[1].set_title('CO Molecule')
plt.show()

### 1.2 Relaxing the Isolated Systems
Before combining them, we must find the lowest energy (relaxed) states of the bare slab and the isolated molecule.

In [ ]:
print("Relaxing bare Pt(111) slab...")
opt_slab = FIRE(slab, trajectory='pt_slab.traj', logfile=None)
opt_slab.run(fmax=0.05)
e_slab = slab.get_potential_energy()
print(f"Energy of bare slab: {e_slab:.2f} eV\n")

print("Relaxing CO molecule...")
opt_mol = FIRE(adsorbate, trajectory='co_mol.traj', logfile=None)
opt_mol.run(fmax=0.05)
e_adsorbate = adsorbate.get_potential_energy()
print(f"Energy of isolated CO: {e_adsorbate:.2f} eV\n")

### 1.3 Placing Adsorbate and Final Relaxation
Now we place the relaxed CO molecule onto the Pt(111) surface and relax the combined system.

In [ ]:
# Copy the relaxed slab
system = slab.copy()

# Add the adsorbate to the 'ontop' site with a height of 3.0 Å
add_adsorbate(system, adsorbate, height=3.0, position='ontop')

# Set calculator and relax
system.calc = calculator
print("Relaxing combined Pt(111) + CO system...")
opt_sys = FIRE(system, trajectory='pt_co_system.traj', logfile=None)
opt_sys.run(fmax=0.05)
e_system = system.get_potential_energy()

print(f"Energy of combined system: {e_system:.2f} eV")

In [ ]:
# Visualize the relaxed combined system
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
plot_atoms(system, ax[0], rotation=('-75x, 45y, 10z'))
ax[0].set_title('Side View')
plot_atoms(system, ax[1], rotation=('90x, 0y, 0z'))
ax[1].set_title('Top View')
plt.show()

In [ ]:
# Calculate Adsorption Energy
e_ads = e_system - (e_slab + e_adsorbate)
print(f"\nCalculated Adsorption Energy (E_ads): {e_ads:.3f} eV")

### 🏋️‍♂️ Exercise 1: Try a Different Adsorption Site
We placed the CO molecule on the 'ontop' site. What happens if we place it on the 'fcc' or 'hcp' hollow site?
1. Copy the code from section 1.3.
2. Change `position='ontop'` to `position='fcc'`.
3. Relax the new system and calculate its adsorption energy.
4. Which site is more stable?

In [ ]:
# Your code for Exercise 1 here:



## Part 2: Diffusion Coefficient via Molecular Dynamics (MD)

Molecular Dynamics simulates the motion of atoms over time at a specific temperature. By tracking how far atoms move from their initial positions, we can calculate the **Mean Squared Displacement (MSD)**.

According to the Einstein relation, the diffusion coefficient $D$ in 3D space is related to the slope of the MSD over time:
$$MSD(t) = \frac{1}{N} \sum_{i=1}^{N} |r_i(t) - r_i(0)|^2$$
$$D = \lim_{t \to \infty} \frac{MSD(t)}{6t}$$

We will simulate Lithium diffusion in a solid-state electrolyte, $\beta$-Li$_3$PS$_4$ (LPS), at a high temperature (to accelerate the process so we can observe it in a short simulation time).

### 2.1 Retrieving Structures from Materials Project
Instead of building complex structures from scratch, we can retrieve them from the **Materials Project (MP)** database using the `mp-api` package.

You need an API key from your [Materials Project Dashboard](https://materialsproject.org/dashboard).

*Note: If you don't provide an API key below, the code will fall back to a simple built-in Li structure so you can still complete the lab.*

In [ ]:
from mp_api.client import MPRester
from pymatgen.io.ase import AseAtomsAdaptor

# 1. Enter your Materials Project API Key here:
api_key = ""  # Leave empty to use the fallback structure

if api_key:
    try:
        # The MP-ID for beta-Li3PS4 is mp-985592
        with MPRester(api_key) as mpr:
            pmg_struct = mpr.get_structure_by_material_id("mp-985592")
            print(f"Successfully fetched: {pmg_struct.composition.reduced_formula}")
            
        # Convert pymatgen Structure to ASE Atoms object
        solid_structure = AseAtomsAdaptor.get_atoms(pmg_struct)
        
        # For LPS, the unit cell has 32 atoms. Let's use a 1x1x2 supercell (64 atoms) for MD
        md_supercell = solid_structure * (1, 1, 2)
        
    except Exception as e:
        print(f"Failed to fetch from MP: {e}")
        print("Falling back to simple Li bcc structure...")
        li_bulk = bulk('Li', 'bcc', cubic=True)
        md_supercell = li_bulk * (3, 3, 3)
else:
    print("No API key provided. Using fallback Li bcc structure...")
    li_bulk = bulk('Li', 'bcc', cubic=True)
    md_supercell = li_bulk * (3, 3, 3)

# Assign calculator and initial velocities
md_supercell.calc = calculator
temperature_K = 1000 # High temperature to accelerate diffusion
MaxwellBoltzmannDistribution(md_supercell, temperature_K * units.kB)

In [ ]:
# Visualize the initial bulk structure
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
plot_atoms(md_supercell, ax, rotation=('-45x, 45y, 0z'))
ax.set_title('MD Supercell')
plt.show()

### 2.2 Running the NVT Simulation
We use the Langevin thermostat to maintain the temperature (NVT ensemble).

In [ ]:
# Set up the MD simulation
time_step = 2.0 * units.fs  # 2 femtoseconds
friction = 0.02 / units.fs

md = Langevin(md_supercell, 
              time_step, 
              temperature_K * units.kB, 
              friction, 
              trajectory='md_run.traj',
              logfile='md_run.log')

# Store positions for MSD calculation
positions = []
times = []

def collect_data():
    positions.append(md_supercell.get_positions())
    times.append(md.get_time() / units.fs) # Time in fs

md.attach(collect_data, interval=10)

# Run simulation for 500 steps (1000 fs = 1 ps)
# Note: In a real scenario, this should run much longer (e.g., 50-100 ps)
print("Starting MD simulation (this may take a few moments)...")
md.run(500)
print("MD simulation finished.")

### 2.3 Calculating Mean Squared Displacement (MSD)
For materials like LPS, we are usually only interested in the diffusion of Lithium ions. We must filter the atoms to track only Lithium before calculating the MSD.

In [ ]:
positions = np.array(positions)
times = np.array(times) / 1000.0  # Convert fs to ps

# Find indices of Lithium atoms
symbols = np.array(md_supercell.get_chemical_symbols())
li_indices = np.where(symbols == 'Li')[0]
print(f"Calculating MSD for {len(li_indices)} Li atoms out of {len(symbols)} total atoms.")

# Initial positions of Li atoms
pos_0_li = positions[0, li_indices, :]

# Calculate MSD: Average over Li atoms for each time step
msd = []
for t_step in range(len(positions)):
    pos_t_li = positions[t_step, li_indices, :]
    diff = pos_t_li - pos_0_li
    sq_dist = np.sum(diff**2, axis=1) # Squared distance for each Li atom
    msd.append(np.mean(sq_dist))

msd = np.array(msd)

### 2.4 Extracting Diffusion Coefficient
We plot MSD vs time. The slope of the linear region is $6D$.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(times, msd, 'b-', linewidth=2, label='MSD')
plt.xlabel('Time (ps)')
plt.ylabel('Mean Squared Displacement (Å$^2$)')
plt.title('Li Diffusion at 1000 K')
plt.grid(True, linestyle='--', alpha=0.7)

# Linear fit to extract diffusion coefficient
# Ignoring the very beginning (ballistic regime) if simulation was longer
slope, intercept = np.polyfit(times, msd, 1)
plt.plot(times, slope*times + intercept, 'r--', label=f'Fit: slope = {slope:.2f}')
plt.legend()
plt.show()

# Unit conversion:
# Slope is in Å^2 / ps = 10^-20 m^2 / 10^-12 s = 10^-8 m^2/s = 10^-4 cm^2/s
diffusion_coeff = slope / 6.0
print(f"Estimated Diffusion Coefficient (D): {diffusion_coeff:.4f} Å^2/ps")
print(f"Or {diffusion_coeff * 1e-4:.2e} cm^2/s")

### 🏋️‍♂️ Exercise 2: Temperature Dependence of Diffusion
Diffusion is a thermally activated process. How does the diffusion coefficient change if we increase or decrease the temperature?
1. Duplicate the MD setup and analysis cells.
2. Change `temperature_K` to 500 K or 1500 K.
3. Re-run the simulation and calculate the new diffusion coefficient.
4. How much did the diffusion coefficient change?

In [ ]:
# Your code for Exercise 2 here:

